# Grad-CAM Compatibility (MobileNetV2 Plant Disease)


**Model:** MobileNetV2 (trained weights loaded from Drive)

**Grad-CAM Target Layer:** `block_16_project`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU detected:" , (gpus[0].name if gpus else 'None'))

In [ ]:
# -----------------------------
# Dataset paths (do not change)
# -----------------------------
DRIVE_DATASET = "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation"
LOCAL_DATASET = "/content/dataset"

# Fast copy to local disk for speed
if not os.path.exists(LOCAL_DATASET):
    print("Copying dataset to local disk...")
    !rm -rf /content/dataset
    !cp -r "/content/drive/MyDrive/Plant_leave_diseases_dataset_with_augmentation" /content/dataset

DATASET_DIR = LOCAL_DATASET

# Hyperparameters (keep same)
IMG_SIZE = (224, 224)
BATCH_SIZE = 24
EPOCHS = 5
LEARNING_RATE = 1e-4

MODEL_SAVE_DIR = "/content/drive/MyDrive/models"
MOB_MODEL_PATH = f"{MODEL_SAVE_DIR}/MobileNetV2.h5"

print("DATASET_DIR:", DATASET_DIR)
print("MOB_MODEL_PATH:", MOB_MODEL_PATH)

In [ ]:
# -----------------------------
# Common Data Pipeline (ONE cell)
# -----------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

print("Number of classes:", num_classes)
print("Class names:", class_names)

## Build Model (Functional API)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2


def build_model(model_name, num_classes, input_shape=(224, 224, 3)):
    # Functional model with MobileNetV2 backbone + identical classification head.
    if model_name != "MobileNetV2":
        raise ValueError("This compatibility notebook is for MobileNetV2 only.")

    inputs = layers.Input(shape=input_shape)

    base_model = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape,
    )
    base_model.trainable = False

    # Keep the same preprocessing used during training.
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


In [ ]:
# Load trained MobileNetV2 model weights
print("Loading trained weights...")
mobile_model = build_model("MobileNetV2", num_classes)
mobile_model.load_weights(MOB_MODEL_PATH)
print("Weights loaded successfully.")

## Grad-CAM (Final Cell)

In [ ]:
# Select last conv layer using the required layer name
TARGET_LAYER_NAME = "block_16_project"

try:
    conv_layer = mobile_model.get_layer(TARGET_LAYER_NAME)
except Exception as e:
    # Helpful debug if the layer name differs in your saved model graph.
    # Still keeps the Grad-CAM cell clean.
    available = [l.name for l in mobile_model.layers]
    raise ValueError(
        f"Could not find layer '{TARGET_LAYER_NAME}' in mobile_model.get_layer()." 
        f"\nError: {e}\n\nTop-level layers available: {available}"
    )

grad_model = tf.keras.models.Model(
    inputs=mobile_model.input,
    outputs=[conv_layer.output, mobile_model.output],
)

# Load one sample image from validation dataset
for batch_images, batch_labels in val_ds.take(1):
    sample_image = batch_images[0].numpy().astype("uint8")
    sample_label = int(batch_labels[0].numpy())
    break

# Prepare image for model input
img_array = tf.expand_dims(tf.cast(sample_image, tf.float32), axis=0)

# Forward pass
preds = grad_model(img_array)[1]
pred_index = int(tf.argmax(preds[0]).numpy())

# Compute gradients of the predicted class score w.r.t. conv feature maps
with tf.GradientTape() as tape:
    conv_outputs, preds = grad_model(img_array)
    loss = preds[:, pred_index]

grads = tape.gradient(loss, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

conv_outputs = conv_outputs[0]
heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
heatmap = tf.maximum(heatmap, 0)
heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
heatmap = heatmap.numpy()

# Overlay heatmap on the original image
heatmap_resized = cv2.resize(heatmap, (IMG_SIZE[1], IMG_SIZE[0]))
heatmap_uint8 = np.uint8(255 * heatmap_resized)
heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

overlay = cv2.addWeighted(
    cv2.cvtColor(sample_image, cv2.COLOR_RGB2BGR), 0.6,
    heatmap_color, 0.4, 0
)
overlay = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

# Display
plt.figure(figsize=(14, 4))
plt.subplot(1, 3, 1)
plt.imshow(sample_image)
plt.title(f"Original\nTrue: {class_names[sample_label]}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(heatmap_resized, cmap="jet")
plt.title(f"Grad-CAM\nPred: {class_names[pred_index]}")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.title("Overlay")
plt.axis("off")

plt.tight_layout()
plt.show()